# 🔺 Cuaderno 02 — Eliminación Gaussiana

**Métodos Numéricos · Unidad 3: Sistemas de Ecuaciones Lineales**

Este cuaderno resuelve $\mathbf{A}\mathbf{x}=\mathbf{b}$ con el método más
antiguo que sigue en uso: **eliminación gaussiana**. Al terminarlo sabrás
convertir un sistema en uno triangular, despejar de abajo hacia arriba, y
—lo más importante— **cuándo el método falla y por qué el pivoteo lo salva**.

### 🎯 Objetivos

- Convertir un sistema en **triangular superior** con operaciones de renglón.
- Programar la eliminación y la sustitución hacia atrás.
- Entender los **multiplicadores** $m_{ik}$ y por qué se dividen entre el pivote.
- Explicar con un ejemplo numérico **por qué el pivoteo parcial es necesario**.
- Calcular el **determinante** como producto de pivotes.
- Estimar el **costo** del método y saber cuándo conviene otro.

### 🧭 Cómo está armado

| § | Contenido |
|---|---|
| 1–4 | el método, paso a paso, con aritmética **exacta** |
| 5 | pivoteo: el pivote cero y el pivote diminuto |
| 6 | determinante y costo |
| 7 | ejercicios para completar |
| 8 | caso de ingeniería: circuito de 3 mallas |
| 9 | preguntas calificables |

> 💡 Todo el cuaderno trabaja con `M.sym[[...]]`, que guarda **enteros y
> fracciones exactas**. Así, cuando el resultado salga $1/2$, verás $1/2$ y no
> $0.4999999999999999$. Es la única forma honesta de estudiar un método directo:
> con flotantes, el error de redondeo se mezcla con el método y ya no sabes qué
> estás viendo.


In [1]:
#@title ⚙️ Cargar entorno (ejecuta PRIMERO)

import os, sys, subprocess, importlib

try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

REPO_URL = "https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales.git"


def _buscar_raiz(base, saltos=6):
    """Sube desde `base` hasta la carpeta que tiene config/config.yaml."""
    b = base
    for _ in range(saltos):
        if os.path.exists(os.path.join(b, "config", "config.yaml")):
            return os.path.abspath(b)
        padre = os.path.dirname(b)
        if padre == b:
            break
        b = padre
    return None


if ES_COLAB:
    # El repo se clona DENTRO de Drive, no en /content (que se borra al cerrar).
    from google.colab import drive
    drive.mount("/content/drive")

    BASE = "/content/drive/MyDrive/CursoMN"
    REPO = os.path.join(BASE, "U3_Repo")
    os.makedirs(BASE, exist_ok=True)
    try:
        if not os.path.exists(os.path.join(REPO, ".git")):
            print("Clonando el repositorio (la primera vez tarda un poco)...")
            subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
        else:
            print("Actualizando el repositorio...")
            subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
        print("✅ Repositorio listo en", REPO)
    except Exception as _exc:
        print("⚠️ No pude clonar/actualizar el repositorio:", _exc)
else:
    _ruta_nb = globals().get("__vsc_ipynb_file__")
    REPO = _buscar_raiz(os.path.dirname(os.path.abspath(_ruta_nb))) if _ruta_nb else None
    REPO = REPO or _buscar_raiz(os.getcwd())

if REPO is None:
    raise RuntimeError(
        "No encuentro la carpeta del repositorio (la que tiene config/config.yaml).\n"
        "Abre el cuaderno desde dentro de U3_Repo o revisa que el clon de Colab exista."
    )

print("Repositorio :", REPO)

# En sys.path van las DOS raíces necesarias:
#   REPO/lib  -> robomat.py y el paquete mn_motor/
#   REPO      -> el paquete preguntas/
for _ruta in (os.path.join(REPO, "lib"), REPO):
    if _ruta not in sys.path:
        sys.path.insert(0, _ruta)

import robomat as _robomat
importlib.reload(_robomat)
from robomat import *               # noqa: F401,F403

# Explícitos, para que Pylance los vea y no dependamos de lo que reexporte
# la biblioteca.
import numpy as np
import sympy as sp

print("✅ Biblioteca:", _robomat.__file__)

# ============================================================
# Motor de calificación
# ============================================================
MODO_MOTOR = "modulos"      # "modulos" (depurar) | "blob" (lo que ve el alumno)
MODO = "blob" if ES_COLAB else MODO_MOTOR

if MODO == "modulos":
    try:
        for _n in [n for n in sys.modules
                   if n.split(".")[0] in ("mn_motor", "preguntas")]:
            del sys.modules[_n]
        import mn_motor as _motor
        from mn_motor import *      # noqa: F401,F403
    except ImportError as _exc:
        print("ℹ️ No pude importar el motor del repositorio:", _exc)
        MODO = "blob"

if MODO == "blob":
    import base64 as _b64
    import zlib as _zlib

    # Cada cuaderno tiene su propio blob: aquí va el del Cuaderno 02.
    ruta_motor = os.path.join(REPO, "grader", "u3_02_gaussiana.txt")
    with open(ruta_motor, encoding="utf-8") as _f:
        _blob = _f.read().strip()
    _codigo = _zlib.decompress(_b64.b64decode(_blob)).decode("utf-8")
    os.environ["U3_REPO"] = REPO
    _MOTOR = {"__name__": "blob_u3_02"}
    exec(_codigo, _MOTOR)
    globals().update({_n: _MOTOR[_n] for _n in _MOTOR["__all__"]})


# Este cuaderno califica SU conjunto de preguntas (carpeta preguntas/u3_02_gaussiana).
# En modo blob ya viene incrustado, pero en modo módulos hay que declararlo.
usar_cuaderno("u3_02_gaussiana", "U3_02_Eliminacion_Gaussiana")

print(f"✅ Motor cargado ({MODO}) · cuaderno {cuaderno()}")
print("   Sigue con la celda de identificación.")

Repositorio : o:\Mi unidad\CursoMN\U3 Sistemas de ecuaciones lineales algebráicas\U3_Repo
✅ Biblioteca: o:\Mi unidad\CursoMN\U3 Sistemas de ecuaciones lineales algebráicas\U3_Repo\lib\robomat.py
✅ Motor cargado (modulos) · cuaderno u3_02_gaussiana
   Sigue con la celda de identificación.


## 0. Tu identificación

Cada alumno recibe **sus** matrices, sembradas con el Número de Control que sale
de tu correo. Copiar la respuesta del compañero da mal.


In [2]:
#@title 👤 Identificación (correo institucional)
if ES_COLAB:
    import requests
    import google.auth
    import google.auth.transport.requests
    from google.colab import auth

    try:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        info = requests.get(
            "https://www.googleapis.com/drive/v3/about?fields=user",
            headers={"Authorization": "Bearer " + creds.token},
            timeout=20,
        ).json()
        cuenta = info.get("user", {})
        email = cuenta.get("emailAddress", "")
        nombre = cuenta.get("displayName", "")
    except Exception as e:
        print("⚠️ No pude obtener tu correo automáticamente.")
        print("   Vuelve a ejecutar y acepta el permiso en la ventana de Google.")
        print("   Detalle:", e)
        email, nombre = "", ""
else:
    email = "m16330887@hermosillo.tecnm.mx"      # <-- cambia por el tuyo
    nombre = ""

if email and email.lower().endswith(".tecnm.mx"):
    alumno_id = email
    print("✅ Identificado:", email)
    print("   Número de control:", extraer_nc(email))
else:
    from IPython.display import HTML, display as _display
    alumno_id = None
    _display(HTML(
        '<p style="color:#b00020;background:#ffe0e0;padding:12px;border-radius:8px;'
        'font-weight:bold">⛔ Necesito tu correo institucional (@….tecnm.mx).<br>'
        'Si iniciaste sesión con otra cuenta, ciérrala o usa una ventana de '
        'incógnita.</p>'))
    print("Correo detectado:", repr(email))

if alumno_id is None:
    raise RuntimeError("Detente: sin correo institucional no puedo generar tu tarea.")

MI_TAREA = generar_tarea(alumno_id, nombre)
print(f"\nListo: {len(MI_TAREA['preguntas'])} preguntas para el NC {MI_TAREA['nc']}.")


✅ Identificado: m16330887@hermosillo.tecnm.mx
   Número de control: 16330887

Listo: 8 preguntas para el NC 16330887.


## 1. Formulación matricial del sistema

### 1.1 Forma estándar: $n$ ecuaciones con $n$ incógnitas

Un sistema lineal de $n$ ecuaciones con $n$ incógnitas se escribe

$$
\begin{aligned}
a_{11}x_1 + a_{12}x_2 + \cdots + a_{1n}x_n &= b_1 \\
a_{21}x_1 + a_{22}x_2 + \cdots + a_{2n}x_n &= b_2 \\
&\vdots \\
a_{n1}x_1 + a_{n2}x_2 + \cdots + a_{nn}x_n &= b_n
\end{aligned}
$$

Cada símbolo tiene un significado preciso:

| Símbolo | Qué es | Cuántos hay |
|---|---|---|
| $n$ | Número de ecuaciones = número de incógnitas (sistema *cuadrado*) | 1 valor |
| $x_1, x_2, \dots, x_n$ | Las **incógnitas**: lo que queremos calcular | $n$ |
| $a_{ij}$ | **Coeficiente**: el número que multiplica a $x_j$ en la ecuación $i$ | $n \times n$ |
| $b_i$ | **Término independiente**: el dato conocido del lado derecho de la ecuación $i$ | $n$ |

El doble subíndice de $a_{ij}$ no es casual: el **primero** dice *en qué
ecuación* está (el renglón) y el **segundo**, *a qué incógnita* acompaña (la
columna). Así, $a_{23}$ es el coeficiente que multiplica a $x_3$ en la segunda
ecuación.

### 1.2 De la forma estándar a la forma matricial

Al escribir las ecuaciones una debajo de otra salta a la vista un patrón: las
incógnitas $x_1, x_2, \dots, x_n$ aparecen **en el mismo orden en todas las
ecuaciones**. Eso permite separar los datos (los números) de las incógnitas
(las letras) y escribir el sistema tres veces más compacto:

1. Se toman los coeficientes de cada ecuación, en orden, y se coloca cada
   ecuación en un renglón. Eso forma la **matriz de coeficientes** $\mathbf{A}$.
2. Las incógnitas se ponen en una sola columna: el **vector de incógnitas**
   $\mathbf{x}$.
3. Los términos independientes se ponen en otra columna: el **vector de términos
   independientes** $\mathbf{b}$.

$$\mathbf{A} =
\begin{bmatrix} a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
a_{n1} & a_{n2} & \cdots & a_{nn} \end{bmatrix},
\qquad
\mathbf{x} = \begin{bmatrix} x_1 \\ x_2 \\ \vdots \\ x_n \end{bmatrix},
\qquad
\mathbf{b} = \begin{bmatrix} b_1 \\ b_2 \\ \vdots \\ b_n \end{bmatrix}$$

de modo que todo el sistema se reduce a

$$\mathbf{A}\mathbf{x} = \mathbf{b}.$$

### 1.3 Por qué la forma matricial dice exactamente lo mismo

No es una abreviación decorativa. Al multiplicar $\mathbf{A}$ por $\mathbf{x}$,
**cada renglón produce una ecuación**:

$$\begin{bmatrix} a_{11} & a_{12} \\ a_{21} & a_{22} \end{bmatrix}
\begin{bmatrix} x_1 \\ x_2 \end{bmatrix}
= \begin{bmatrix} a_{11}x_1 + a_{12}x_2 \\ a_{21}x_1 + a_{22}x_2 \end{bmatrix}$$

El primer renglón del resultado es la primera ecuación y el segundo es la
segunda. En general, el renglón $i$ del producto es

$$a_{i1}x_1 + a_{i2}x_2 + \cdots + a_{in}x_n,$$

que es exactamente el lado izquierdo de la ecuación $i$. Por eso resolver
$\mathbf{A}\mathbf{x} = \mathbf{b}$ y resolver el sistema escrito a mano son **el
mismo problema**: se puede trabajar con la matriz sin perder información.

### 1.4 La matriz aumentada

Para no arrastrar a $\mathbf{b}$ por separado, se apila como última columna y se
separa con una raya vertical:

$$[\mathbf{A}\mid\mathbf{b}] =
\begin{bmatrix} a_{11} & a_{12} & \cdots & a_{1n} & \mid & b_1 \\
\vdots & \vdots & \ddots & \vdots & \mid & \vdots \\
a_{n1} & a_{n2} & \cdots & a_{nn} & \mid & b_n \end{bmatrix}$$

Cada operación que se haga sobre los coeficientes de un renglón se hace también
sobre su $b_i$, y tenerlos juntos evita olvidarlo.

### 1.5 Ejemplo con enteros: el sistema de todo el cuaderno

$$\begin{aligned} 
 2x_1 +  x_2 -  x_3 &=   8 \\ 
-3x_1 -  x_2 + 2x_3 &= -11 \\ 
-2x_1 +  x_2 + 2x_3 &=  -3 
\end{aligned}$$

Al extraer los coeficientes de cada incógnita y los términos independientes, el
sistema se escribe en forma matricial $\mathbf{A}\mathbf{x} = \mathbf{b}$:

$$\underbrace{\begin{bmatrix} 2x_1 + x_2 - x_3 \\ -3x_1 - x_2 + 2x_3 \\ -2x_1 + x_2 + 2x_3 \end{bmatrix}}_{\mathbf{A}\mathbf{x}} = \underbrace{\begin{bmatrix} 2 & 1 & -1 \\ -3 & -1 & 2 \\ -2 & 1 & 2 \end{bmatrix}}_{\mathbf{A}} \underbrace{\begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix}}_{\mathbf{x}} = \underbrace{\begin{bmatrix} 8 \\ -11 \\ -3 \end{bmatrix}}_{\mathbf{b}}$$

Entonces definimos su matriz aumentada $[\mathbf{A}|\mathbf{b}]$:

In [3]:
# M.sym guarda los números exactos: enteros y fracciones, no flotantes.
A = M.sym[[ 2,  1, -1],
          [-3, -1,  2],
          [-2,  1,  2]]
b = M.sym[[  8], 
          [-11], 
          [ -3]]
Ab = M.sym[[A, b]]        # matriz aumentada, al estilo de MATLAB

mostrar(Ab)
print("Forma:", Ab.shape)


<IPython.core.display.Math object>

Forma: (3, 4)


### 1.6 ¿Por qué no basta con despejar?

Consideremos el siguiente sistema $2 \times 2$ y su matriz aumentada $[\mathbf{A}|\mathbf{b}]$:

$$\begin{aligned} 
2x_1 +  x_2 &= 4 \\ 
 x_1 + 3x_2 &= 7 
\end{aligned}
\quad \Longleftrightarrow \quad
\underbrace{
  \begin{bmatrix} 
  2 & 1 & \big| & 4 \\ 
  1 & 3 & \big| & 7 
  \end{bmatrix}
}_{[\mathbf{A}|\mathbf{b}]\;(2\times 3)}$$

Si intentamos despejar $x_1$ directamente de cada renglón por separado:

- **Renglón 1:** $2x_1 + x_2 = 4 \implies x_1 = \frac{4 - x_2}{2}$
- **Renglón 2:** $x_1 + 3x_2 = 7 \implies x_1 = 7 - 3x_2$

Como se observa, **ninguna ecuación se puede resolver de manera aislada** porque $x_1$ sigue dependiendo de la variable desconocida $x_2$. 

Para resolver el sistema es necesario combinar las ecuaciones (o renglones) para eliminar una de las incógnitas, que es precisamente lo que realizan las operaciones elementales de renglón en la eliminación gaussiana.

## 2. Eliminación hacia adelante (*forward elimination*)

El objetivo de esta fase es **quitar incógnitas de las ecuaciones de abajo**
hasta que la última tenga una sola. La herramienta son las **operaciones
elementales de renglón**:

$$R_i \leftarrow R_i - m_{ik}R_k$$

### 2.1 La idea, en palabras

En álgebra elemental un sistema se resuelve **eliminando** incógnitas: se toma
una ecuación, se multiplica por un número conveniente y se le resta a otra para
que una incógnita desaparezca. Con dos ecuaciones,

$$
\begin{aligned}
a_{11}x_1 + a_{12}x_2 &= b_1 &&\text{(ecuación 1)}\\
a_{21}x_1 + a_{22}x_2 &= b_2 &&\text{(ecuación 2)}
\end{aligned}
$$

si se multiplica la ecuación 1 por $\dfrac{a_{21}}{a_{11}}$ y se le resta a la
ecuación 2, la incógnita $x_1$ desaparece de la segunda:

$$\left(a_{21} - \frac{a_{21}}{a_{11}}a_{11}\right)x_1 +
\left(a_{22} - \frac{a_{21}}{a_{11}}a_{12}\right)x_2
= b_2 - \frac{a_{21}}{a_{11}}b_1,$$

es decir

$$0\cdot x_1 + a_{22}'\,x_2 = b_2',
\qquad a_{22}' = a_{22} - \frac{a_{21}}{a_{11}}a_{12},
\qquad b_2' = b_2 - \frac{a_{21}}{a_{11}}b_1.$$

La ecuación 2 quedó con **una sola incógnita** y ya se puede despejar.

### 2.2 Por qué esa resta es legal

Multiplicar una ecuación por una constante y restársela a otra **no cambia la
solución**. Si un vector $\mathbf{x}$ cumple las dos ecuaciones originales,
también cumple cualquier combinación de ellas, y viceversa: se reemplaza una
ecuación por otra equivalente, sin perder información. Es el mismo argumento
que se usa en álgebra elemental.

### 2.3 El pivote y el multiplicador

El número entre el que se divide ($a_{11}$ en el ejemplo) se llama **pivote**: es
el coeficiente de la incógnita que se va a eliminar. El factor que se resta se
llama **multiplicador**:

$$m_{ik} = \frac{a_{ik}^{(k)}}{a_{kk}^{(k)}}.$$

El subíndice $k$ dice **qué incógnita** se elimina en ese paso y la $i$, **de qué
renglón** se elimina. El pivote es entonces $a_{kk}$, el elemento de la diagonal
del renglón $k$, y debe cumplir $a_{kk} \neq 0$ porque está en un denominador.

#### Cómo leer el superíndice

> 📖 El superíndice **no es una potencia**: es el **número de paso**.
> $a_{22}^{(2)}$ se lee «$a_{22}$ con el valor que tiene en el paso 2».

En $a_{kk}^{(k)}$ las tres $k$ son renglón, columna y paso; coinciden porque el
método recorre la diagonal. El superíndice hace falta porque **el valor cambia**:
en este sistema $a_{22}$ vale $-1$ al empezar y $1/2$ después del primer paso.
Por eso se escribe

$$m_{32} = \frac{a_{32}^{(2)}}{a_{22}^{(2)}} = \frac{2}{1/2} = 4,$$

y no $1/(-1) = -1$, que saldría de usar los valores viejos.

Y ojo con el orden: en $m_{ik}$, $i$ es el renglón que **se modifica** y $k$ el
renglón **pivote**. Es $m_{ik}$, no $m_{ki}$.

### 2.4 El algoritmo con $n$ ecuaciones

El mismo paso se repite para cada columna $k = 1, 2, \dots, n-1$:

1. **Tomar el pivote** $a_{kk}^{(k)}$ del renglón $k$.
2. Para cada renglón de abajo, $i = k+1, \dots, n$, calcular su multiplicador
   $m_{ik} = a_{ik}^{(k)}/a_{kk}^{(k)}$.
3. **Restar** $m_{ik}$ veces el renglón $k$ al renglón $i$, en **todos** los
   coeficientes desde la columna $k$ hasta la $n$, y también en $\mathbf{b}$:

$$a_{ij}^{(k+1)} = a_{ij}^{(k)} - m_{ik}\,a_{kj}^{(k)},
\qquad b_i^{(k+1)} = b_i^{(k)} - m_{ik}\,b_k^{(k)}.$$

El superíndice $(k)$ significa «el valor que tiene en el paso $k$»: los números
van cambiando de un paso al siguiente.

### 2.5 Qué se logra: forma triangular

Al terminar, todo lo que está **debajo de la diagonal** quedó en cero:

$$[\mathbf{U}\mid\mathbf{d}] = \begin{bmatrix}
u_{11} & u_{12} & \cdots & u_{1n} & \mid & d_1 \\
0 & u_{22} & \cdots & u_{2n} & \mid & d_2 \\
\vdots & \vdots & \ddots & \vdots & \mid & \vdots \\
0 & 0 & \cdots & u_{nn} & \mid & d_n
\end{bmatrix}$$

Eso es lo que se buscaba, porque la **última** ecuación ya tiene una sola
incógnita ($u_{nn}x_n = d_n$). El método completo es, entonces, **triangular
primero y despejar después**.

> ⚠️ En esta sección se supone que **todos los pivotes son distintos de cero**.
> Si alguno vale cero o es muy pequeño, hace falta lo de la §5 (pivoteo).

### 2.6 Ejemplo algebraico $2\times 2$

En un sistema $2\times 2$ hay un solo paso. El pivote es $a_{11}$ y el
multiplicador $m_{21} = a_{21}/a_{11}$. Al restar $R_2 \leftarrow R_2 - m_{21}R_1$
la matriz aumentada pasa de

$$
\begin{bmatrix} a_{11} & a_{12} & \mid & b_1 \\ a_{21} & a_{22} & \mid & b_2 \end{bmatrix}
\quad\longrightarrow\quad
\begin{bmatrix} a_{11} & a_{12} & \mid & b_1 \\ 0 & a_{22}' & \mid & b_2' \end{bmatrix},
$$

con $a_{22}' = a_{22} - m_{21}a_{12}$ y $b_2' = b_2 - m_{21}b_1$.

### 2.7 Paso $k=1$ en nuestro sistema de $3\times 3$

Con $a_{11}=2$:

$$m_{21} = \frac{-3}{2}, \qquad m_{31} = \frac{-2}{2} = -1$$

> 🔢 **Del índice del libro al índice de Python.** La teoría numera desde 1
> (la convención de los libros y de MATLAB) y Python desde 0. La regla es una
> sola: **resta 1 a cada índice al escribirlo en el código**; y al revés, los
> mensajes del código vuelven a sumar 1 para hablar en el idioma del alumno.

| Libro | Python | En nuestro sistema |
|---|---|---|
| $a_{11}$ | `Ab[0, 0]` | pivote del paso 1 |
| $a_{22}$ | `Ab[1, 1]` | pivote del paso 2 |
| $a_{ij}$ | `Ab[i-1, j-1]` | — |
| paso $k$ ($1 \dots n-1$) | `k` de `0` a `n-2` | — |

El desfase no se puede evitar: la notación matemática es 1-based y Python
0-based, y LAPACK y MATLAB también documentan en 1-based. Cambiarla aquí haría
que el cuaderno no coincidiera con ningún libro ni con el examen. Lo que sí se
puede —y se hace— es **no esconder el cambio**: en las celdas siguientes cada
índice lleva su comentario.

In [6]:
Ab = M.sym[[A, b]]

# Paso k = 1: eliminar x_1 de los renglones 2 y 3.
# El libro cuenta los renglones 1, 2, 3; Python los cuenta 0, 1, 2.
for renglon in (1, 2):                      # renglones 2 y 3 del libro
    pivote = Ab[0, 0]                       # a_11^(1)   libro (1,1) -> Ab[0,0]
    m = Ab[renglon, 0] / pivote             # m_i1 = a_i1^(1) / a_11^(1)
    print(f"m_{renglon+1},1 = {Ab[renglon,0]} / {pivote} = {m}")
    Ab[renglon, :] = Ab[renglon, :] - m * Ab[0, :]    # R_i <- R_i - m R_1

print("\nDespués del paso k=1:")
mostrar(Ab)

m_2,1 = -3 / 2 = -3/2
m_3,1 = -2 / 2 = -1

Después del paso k=1:


<IPython.core.display.Math object>

Los dos ceros de la primera columna ya están. Ahora **el renglón 1 no se
vuelve a tocar**: la incógnita $x_1$ ya vive solo ahí.

### 2.8 Paso $k=2$: eliminar $x_2$ del renglón 3

El pivote ahora es $a_{22} = 1/2$, **no** el $a_{22}$ original: la eliminación del
paso anterior modificó los coeficientes de abajo. Por eso el pivote se lee del
estado actual de la matriz (el que lleva el superíndice $(k)$), nunca del
sistema original. Es el error más común al programar el método.


In [ ]:
# Paso k = 2: ahora el pivote es a_22, que en Python es Ab[1, 1]
pivote = Ab[1, 1]           # a_22^(2) = 1/2   libro (2,2) -> Ab[1,1]
m32 = Ab[2, 1] / pivote     # m_32 = a_32^(2) / a_22^(2)
print(f"m_32 = {Ab[2,1]} / {pivote} = {m32}")

# R_3 <- R_3 - m_32 R_2     (R_3 = Ab[2], R_2 = Ab[1])
Ab[2, :] = Ab[2, :] - m32 * Ab[1, :]
print("\nDespués del paso k=2:")
mostrar(Ab)


m_32 = 2 / 1/2 = 4

Después del paso k=2:


<IPython.core.display.Math object>

Ya está triangular. Comprueba el patrón: debajo de la diagonal, puros ceros, y
la tercera ecuación quedó con una sola incógnita.

### 2.9 La eliminación, como función

El patrón de arriba se repite para cada columna. Encerrarlo en una función
permite reusarlo y, sobre todo, **probarlo**. Obsérvese que la función respeta
literalmente el algoritmo de §2.4: dos ciclos, `k` sobre las columnas e `i`
sobre los renglones de abajo, y el pivote leído de `Ab[k, k]` en cada paso.


In [ ]:
def gauss_eliminar(A, b):
    """Devuelve la matriz aumentada triangularizada, SIN pivoteo.

    A y b deben ser exactas (M.sym) para que no haya error de redondeo.
    """
    Ab = M.sym[[A, b]]                      # copia: no toca los argumentos
    n = Ab.rows
    # k e i son índices de PYTHON. El paso k+1 del libro usa el pivote Ab[k,k].
    for k in range(n - 1):                  # columna del pivote
        for i in range(k + 1, n):           # renglones de abajo
            if Ab[k, k] == 0:
                raise ZeroDivisionError(
                    f"pivote cero en la columna {k+1}: hace falta pivoteo")
            m = Ab[i, k] / Ab[k, k]
            Ab[i, :] = Ab[i, :] - m * Ab[k, :]
    return Ab


mostrar(gauss_eliminar(A, b))


In [ ]:
    # ¿Coincide con lo que hicimos a mano? (Ab = matriz tras los dos pasos)
caso = gauss_eliminar(A, b)
print("¿Igual a mano?", caso == Ab)


## 3. Sustitución hacia atrás (*back substitution*)

Con la matriz triangular superior, se despeja **de abajo hacia arriba**.

### 3.1 Las dos fórmulas

**1. La última incógnita.** Del renglón $n$:

$$u_{nn}x_n = d_n \qquad\Longrightarrow\qquad x_n = \frac{d_n}{u_{nn}}.$$

Requiere $u_{nn} \neq 0$; si el pivoteo se hizo bien, lo es.

**2. Las demás, de abajo hacia arriba.** Para $i = n-1, n-2, \dots, 1$:

$$x_i = \frac{d_i - \displaystyle\sum_{j=i+1}^{n} u_{ij}x_j}{u_{ii}}.$$

La suma $\sum_{j=i+1}^{n}$ recorre **solo las incógnitas a la derecha de la
diagonal** en el renglón $i$, que son justamente las que ya se calcularon. Por eso
en cada paso no queda ninguna incógnita desconocida.

En el renglón $n-1$ la suma tiene **un solo término**:

$$x_{n-1} = \frac{d_{n-1} - u_{n-1,\,n}\,x_n}{u_{n-1,\,n-1}}.$$

En el renglón $n-2$ tiene dos, y así hasta el primero, donde la suma recorre
todas las incógnitas ya conocidas.

### 3.2 Por qué no se puede empezar por arriba

Si se intentara despejar $x_1$ del primer renglón, aparecerían $x_2, \dots, x_n$:
un problema con $n-1$ incógnitas, es decir, **el mismo problema otra vez**. Solo
al empezar por abajo se tiene una incógnita por ecuación.

### 3.3 Costo

Esta fase requiere del orden de $n^2$ operaciones, mucho menos que las
$\tfrac{2}{3}n^3$ de la eliminación. La cuenta exacta está en la §6.

### 3.4 Ejemplo algebraico $2\times 2$

Para un sistema ya triangular,

$$
\begin{bmatrix} a_{11} & a_{12} & \mid & b_1 \\ 0 & a_{22}' & \mid & b_2' \end{bmatrix},
$$

se aplica la fórmula dos veces, de abajo hacia arriba:

$$x_2 = \frac{b_2'}{a_{22}'}, \qquad x_1 = \frac{b_1 - a_{12}x_2}{a_{11}}.$$

La primera división exige $a_{22}' \neq 0$ y la segunda, $a_{11} \neq 0$.

### 3.5 Aplicado a nuestro sistema


In [ ]:
def sustitucion_atras(Ab):
    """Despeja x de una matriz aumentada triangular superior."""
    n = Ab.rows
    x = [None] * n
    # i es índice de PYTHON: el renglón i+1 del libro.
    for i in range(n - 1, -1, -1):          # de la última fila a la primera
        suma = sum(Ab[i, j] * x[j] for j in range(i + 1, n))
        if Ab[i, i] == 0:
            raise ZeroDivisionError(f"pivote cero en la fila {i+1}")
        x[i] = (Ab[i, n] - suma) / Ab[i, i]
    return M.sym[[[v] for v in x]]


x = sustitucion_atras(Ab)
print("x =")
mostrar(x)


In [ ]:
# Contra la respuesta de la biblioteca: deben ser idénticas.
esperado = A.solve(b)
print("nuestra x  :", x.T.tolist())
print("A.solve(b) :", esperado.T.tolist())
print("¿Idénticas?:", x == esperado)

# Y el residual tiene que ser exactamente cero: A@x - b = 0
mostrar(A @ x - b)
print("¿Residual exactamente cero?", (A @ x - b).is_zero_matrix)


## 4. El método completo

Juntando las dos piezas queda el resolvedor. Nótese la **verificación** que se
hace al final: resolver no es solo obtener números, es comprobar que cumplen el
sistema original. Es un hábito que conviene adoptar desde ahora.


In [ ]:
def gauss_solve(A, b):
    """Resuelve A x = b por eliminación gaussiana (sin pivoteo). Devuelve x."""
    Ab = gauss_eliminar(A, b)
    x = sustitucion_atras(Ab)
    # Comprobación: si el residual no es cero, algo salió mal.
    if not (A @ x - b).is_zero_matrix:
        raise ArithmeticError("el residual no es cero: revisa la eliminación")
    return x


x = gauss_solve(A, b)
mostrar(x)
print("¿Exacta toda la cuenta?", es_exacto(x))


### 4.1 ¿Y si el sistema no tiene solución única?

La eliminación **no necesita** saber si hay solución para avanzar: si aparece un
pivote cero, es que $\mathbf{A}$ no es invertible. O el sistema no tiene
solución, o tiene infinitas. El método se detiene; distinguir cuál de los dos
casos es, es tema de otro curso (rango). Aquí basta saber que **un pivote cero
es una señal**, no un accidente.


## 5. Pivoteo parcial: por qué el método se rompe y cómo se arregla

El algoritmo de §2 supone que **todos los pivotes son distintos de cero**. Esa
suposición falla de dos maneras, y la segunda es la peligrosa.

### 5.1 Falla 1: un pivote igual a cero

$$\begin{bmatrix} 0 & 1 & \mid & 1 \\ 1 & 1 & \mid & 3 \end{bmatrix}$$

El sistema tiene solución única ($x_1=2$, $x_2=1$), pero $m_{21} = 1/0$ **no
existe**. El método se cae aunque el problema esté perfectamente bien planteado:
el determinante vale $-1$, así que la solución existe y es única.

### 5.2 Falla 2: un pivote muy pequeño (pero distinto de cero)

Este caso es **más peligroso** porque no da error: entrega un resultado creíble
y equivocado. Sea

$$
\begin{aligned}
10^{-18}\,x_1 + x_2 &= 1\\
x_1 + x_2 &= 2
\end{aligned}
$$

cuya solución verdadera es $x_1 = 1$, $x_2 = 1$. El pivote es $a_{11} = 10^{-18}$,
pequeño pero válido, así que el método procede:

$$m_{21} = \frac{1}{10^{-18}} = 10^{18}.$$

Un multiplicador **un billón de veces más grande** que los datos originales. Al
restar $R_2 \leftarrow R_2 - 10^{18}R_1$:

- coeficiente de $x_2$: $1 - 10^{18}(1) = -10^{18}$
- lado derecho: $2 - 10^{18}(1) = -10^{18}$

De ahí $x_2 = \dfrac{-10^{18}}{-10^{18}} = 1$, que sale bien. Pero al despejar
$x_1$:

$$x_1 = \frac{1 - x_2}{10^{-18}} = \frac{1 - 1}{10^{-18}} = \frac{0}{10^{-18}} = 0,$$

cuando debía dar $x_1 = 1$. **El resultado está completamente mal.**

El motivo es que al calcular $1 - x_2$ se restan dos números casi iguales: el
valor verdadero de la diferencia es del orden de $10^{-18}$, más pequeño que la
precisión con que se representan esos números. Se pierden **todos** los dígitos
significativos y la resta devuelve cero; dividir ese cero no recupera nada. Este
fenómeno se llama **cancelación catastrófica**.

Veamos los dos caminos, exacto y flotante:


In [ ]:
P = M.sym[[0, 1], [1, 1]]
pb = M.sym[[1], [3]]
print("det =", P.det(), "(≠ 0, así que SÍ hay solución única)")

try:
    gauss_solve(P, pb)
except ZeroDivisionError as exc:
    print("⛔", exc)

# Con un intercambio de renglones se arregla. Ojo: P[1,:] es una MATRIZ (1x2)
# y M.sym[[matriz, matriz]] las concatena; para apilar renglones hacen falta
# listas, así que se pasan con .tolist().
P2 = M.sym[[P.tolist()[1], P.tolist()[0]]]
b2 = M.sym[[[pb[1, 0]], [pb[0, 0]]]]
print("\ntras intercambiar renglones:")
mostrar(M.sym[[P2, b2]])
print("x =", gauss_solve(P2, b2).T.tolist())


### 5.3 La estrategia: elegir el mayor de la columna

El **pivoteo parcial** consiste en no aceptar el pivote que toca por posición,
sino **elegir el de mayor magnitud** de la columna. Antes de eliminar en el paso
$k$ se busca

$$p = \arg\max_{i = k,\, k+1,\, \dots,\, n} \left| a_{ik}^{(k)} \right|,$$

es decir, el renglón $p$ donde el coeficiente de la columna $k$ es el más grande
en valor absoluto. Si $p \neq k$, se intercambian los renglones completos:

$$R_k \longleftrightarrow R_p,$$

incluyendo el lado derecho $b$. Recién entonces se calculan los multiplicadores.

### 5.4 Por qué esto resuelve el problema

Después del intercambio, el pivote es el mayor de su columna, así que para
cualquier otro renglón

$$|m_{ik}| = \left| \frac{a_{ik}}{a_{kk}} \right| \le 1.$$

Los multiplicadores ya **no amplifican** nada: restar $m_{ik}$ veces un renglón
resta una cantidad del mismo orden de magnitud que el renglón mismo, así que los
errores de redondeo no se multiplican.

Además, el intercambio garantiza que el pivote no sea cero, salvo que la columna
entera sea cero; en ese caso $\mathbf{A}$ no es invertible y el sistema no tiene
solución única. Es decir: **el pivoteo también diagnostica** los casos patológicos.

> 💡 En la práctica **casi todos los programas hacen pivoteo siempre**, aunque el
> pivote parezca aceptable. Cuesta muy poco y evita los dos problemas a la vez.

### 5.5 Efecto sobre el determinante

Cada intercambio de renglones **cambia el signo** del determinante, pero no su
valor absoluto. Por eso, contando los intercambios $p_{\text{swaps}}$:

$$\det(\mathbf{A}) = (-1)^{p_{\text{swaps}}} \prod_{k=1}^{n} u_{kk}.$$

Es la única corrección que hay que hacerle a la fórmula de §6.1 cuando se usa
pivoteo.


In [ ]:
eps = sp.Rational(1, 10**18)      # sp viene de robomat (sympy reexportado)
Ae = M.sym[[eps, 1], [1, 1]]
be = M.sym[[1], [2]]

print("EXACTO  ->", gauss_solve(Ae, be).T.tolist())

# Lo mismo, pero con flotantes (como lo haría NumPy)
Ae_f = np.array([[1e-18, 1.0], [1.0, 1.0]])
be_f = np.array([1.0, 2.0])
print("FLOAT   ->", np.linalg.solve(Ae_f, be_f).round(6).tolist())


Con flotantes, $x_1$ sale $\approx 111$ en vez de $1$: el error no viene de
"pocos dígitos", viene de dividir entre un pivote casi cero, que amplifica
cualquier error de redondeo por un factor de $10^{18}$.

Con pivoteo parcial se intercambian los renglones, el pivote pasa a ser $1$ y el
resultado es el correcto. El código lo comprueba:


In [ ]:
def gauss_pivoteo(A, b):
    """Igual que gauss_solve, pero elige el pivote mayor de cada columna.

    Versión NUMÉRICA (float): aquí el pivoteo tiene sentido porque los números
    se pueden comparar por magnitud. Con símbolos no siempre se puede.
    """
    Ab = np.array(M[[A, b]], dtype=float) if not isinstance(A, np.ndarray) else \
         np.hstack([np.asarray(A, float), np.asarray(b, float).reshape(-1, 1)])
    n = Ab.shape[0]

    for k in range(n - 1):
        p = k + np.argmax(np.abs(Ab[k:, k]))     # renglón con el pivote mayor
        if p != k:
            Ab[[k, p], :] = Ab[[p, k], :]        # intercambio de renglones
        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i, :] = Ab[i, :] - m * Ab[k, :]

    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (Ab[i, -1] - Ab[i, i+1:n] @ x[i+1:n]) / Ab[i, i]
    return x


print("FLOAT con pivoteo ->", gauss_pivoteo(Ae_f, be_f).round(6).tolist())
print("             (la respuesta correcta es [1.0, 1.0])")


### 5.6 La consecuencia para este cuaderno

Con aritmética **exacta** no hay error de redondeo que amplificar, así que el
pivote diminuto no molesta: el pivote cero sí, porque la división es imposible.
Por eso el material hace las dos cosas:

| Versión | Para qué | Qué enseña |
|---|---|---|
| **exacta** (`M.sym`, sin pivoteo) | verificar el algoritmo | el método es correcto |
| **flotante** (NumPy, con pivoteo) | resolver de verdad | el redondeo existe y el pivoteo lo controla |

Un curso que solo hiciera la primera te dejaría creyendo que los flotantes son
inofensivos. Uno que solo hiciera la segunda te dejaría sin saber si el error que
ves es del método o del redondeo.


## 6. Determinante y costo

### 6.1 El determinante sale casi gratis

Las operaciones de renglón del tipo $R_i \leftarrow R_i - mR_k$ **no cambian** el
determinante, porque sumar a un renglón un múltiplo de otro es una operación que
preserva el valor. Los intercambios sí lo cambian: cada uno lo multiplica por
$-1$ (esa es la razón del factor $(-1)^{p_{\text{swaps}}}$ de §5.5).

Como la matriz triangular tiene por determinante el producto de su diagonal,
queda

$$\det(\mathbf{A}) = (-1)^{p}\prod_{k=1}^{n} u_{kk}$$

con $p$ el número de intercambios. O sea: **ya que triangulizaste, el
determinante es multiplicar la diagonal**. No hay que volver a hacer cuentas.

Esto tiene una consecuencia práctica: calcular $\det(\mathbf{A})$ por el método
de cofactores cuesta $O(n!)$, mientras que por eliminación cuesta lo mismo que
resolver el sistema, $O(n^3)$. Para $n = 20$, la diferencia es astronómica.


In [ ]:
Ab_tri = gauss_eliminar(A, b)
producto = sp.prod([Ab_tri[k, k] for k in range(Ab_tri.rows)])
print("producto de los pivotes:", producto)
print("A.det()                :", A.det())
print("¿Iguales?", producto == A.det())

# Con el circuito del caso de ingeniería (más grande)
print("\nCaso del circuito:", R.det() if 'R' in dir() else "(se define en la §8)")


### 6.2 Qué es un FLOP y para qué contarlos

Un **FLOP** (*floating point operation*) es una operación aritmética entre
números reales: una suma, una resta, una multiplicación o una división. Contarlos
sirve para dos cosas:

1. **Comparar algoritmos** sin depender de la computadora: el que tenga menos
   FLOPs será más rápido.
2. **Predecir cómo crece el tiempo** al aumentar el tamaño $n$ del sistema.

Convención: cada multiplicación, división, suma o resta cuenta como **1 FLOP**.
Una operación combinada del tipo $a - m\cdot c$ cuenta como **2 FLOPs** (una
multiplicación y una resta).

### 6.3 Cuánto cuesta la eliminación

En el paso $k$ hay $n-k$ renglones por eliminar. Cada uno cuesta:

| Concepto | FLOPs |
|---|---|
| Calcular el multiplicador $m_{ik} = a_{ik}/a_{kk}$ | $1$ |
| Actualizar los coeficientes de las columnas $k+1$ a $n$ ($n-k$ productos y $n-k$ restas) | $2(n-k)$ |
| Actualizar el lado derecho $b_i$ (un producto y una resta) | $2$ |
| **Total por renglón** | $2(n-k)+3$ |

Multiplicando por los $n-k$ renglones y sumando sobre todos los pasos:

$$C_{\text{elim}} = \sum_{k=1}^{n-1}(n-k)\left[2(n-k)+3\right]
= \frac{n(n-1)(4n+7)}{6}.$$

### 6.4 Cuánto cuesta la sustitución hacia atrás

La última incógnita cuesta **1** FLOP (una división). Cada una de las demás
cuesta $2(n-i)$ FLOPs por los productos y restas de la suma, más $1$ por la
división final, es decir $2(n-i)+1$. Sumando:

$$C_{\text{sust}} = 1 + \sum_{i=1}^{n-1}\left[2(n-i)+1\right] = n^{2}.$$

Sencillo y exacto: la sustitución cuesta **exactamente** $n^2$ FLOPs.

### 6.5 Por qué el término cúbico es el que manda

Abriendo el numerador de $C_{\text{elim}}$,

$$C_{\text{elim}} = \frac{4n^{3} + 3n^{2} - 7n}{6}
= \frac{2}{3}n^{3} + \frac{1}{2}n^{2} - \frac{7}{6}n,$$

se ve de dónde salen los dos términos. Para $n$ grande el cúbico $\frac{2}{3}n^3$
opaca al cuadrático, mientras que la sustitución se queda en $n^{2}$. El cociente
entre ambos crece sin límite, así que el costo total es $O(n^{3})$ y lo aporta
casi todo la eliminación.

La consecuencia práctica: **duplicar** $n$ multiplica el trabajo de eliminación
por $\approx 8$ (porque $2^3 = 8$), mientras que la sustitución solo se multiplica
por $4$. Por eso, cuando $n$ crece, lo único que importa es reducir las
operaciones de la eliminación: ahí está el interés de los métodos especializados
(LU, Thomas, Cholesky) de los cuadernos siguientes.

### 6.6 La cuenta a mano, con $n = 3$

Usando el mismo sistema de la §2:

1. **Paso $k=1$:** hay $n-k = 2$ renglones por eliminar. Cada uno cuesta
   $2(2)+3 = 7$ FLOPs (1 del multiplicador, $2\cdot2 = 4$ de los coeficientes y 2
   del lado derecho). Total: $2 \times 7 = 14$ FLOPs.
2. **Paso $k=2$:** queda $n-k = 1$ renglón, que cuesta $2(1)+3 = 5$ FLOPs.
3. **Eliminación completa:** $14 + 5 = 19$ FLOPs. La fórmula da
   $\frac{3\cdot2\cdot19}{6} = 19$ ✓
4. **Sustitución hacia atrás:** $x_3$ cuesta $1$; $x_2$, $2(1)+1 = 3$; $x_1$,
   $2(2)+1 = 5$. Total: $9 = n^{2}$ ✓
5. **Total del método:** $19 + 9 = 28$ FLOPs.
6. **Comparación con la cota:** $\frac{2}{3}n^{3} = 18$ frente a los $19$ reales;
   la diferencia viene del término cuadrático que la cota desprecia. Para $n = 3$
   pesa, pero para $n$ grande deja de importar.

La siguiente celda lo confirma con números, y muestra cómo crece:

In [ ]:
print(f"{'n':>6} {'eliminación':>14} {'sustitución':>13} {'total':>14}")
for n in (3, 10, 100, 1000):
    e = sp.Rational(2,3)*n**3 + sp.Rational(1,2)*n**2 - sp.Rational(7,6)*n
    s = n**2
    print(f"{n:>6} {str(e):>14} {s:>13} {str(e+s):>14}")
print("\nDe n=100 a n=1000 el trabajo se multiplica por ~1000 (el cubo).")


## 7. Tu turno

Completa las funciones. Las celdas de abajo las prueban solas: si algo está mal,
te dicen **qué** está mal, no solo que está mal.


In [ ]:
# ✏️ EJERCICIO 1 — completa la eliminación hacia adelante
def mi_eliminacion(A, b):
    """Devuelve la matriz aumentada triangular superior, sin pivoteo.

    Pista: recorre las columnas k = 0 .. n-2 (índices de PYTHON). Para cada
    una, y para cada renglón i = k+1 .. n-1, calcula m = Ab[i,k] / Ab[k,k] y
    resta m veces el renglón k: Ab[i,:] = Ab[i,:] - m * Ab[k,:].
    Ojo: el paso k+1 del libro usa el pivote Ab[k,k].
    """
    Ab = M.sym[[A, b]]
    n = Ab.rows
    # TODO: escribe aquí los dos ciclos
    return None


resultado = mi_eliminacion(A, b)
print("resultado:", "todavía no" if resultado is None else "listo")


In [ ]:
# ✅ Verificación del ejercicio 1 (no modificar)
if resultado is None:
    print("⏳ El ejercicio 1 sigue pendiente.")
else:
    if verificar(resultado, Ab):
        print("✅ Ejercicio 1 correcto")
        print("   Además es exacto:", es_exacto(resultado))
    else:
        print("❌ Ejercicio 1 incorrecto")
        print("   Se esperaba:")
        mostrar(Ab)
        if hasattr(resultado, "shape") and resultado.shape != Ab.shape:
            print(f"   Tu resultado tiene forma {resultado.shape}, se esperaba {Ab.shape}")
        else:
            print("   Revisa los índices: el pivote de la columna k está en Ab[k,k],")
            print("   y solo se modifican los renglones i > k.")


In [ ]:
# ✏️ EJERCICIO 2 — completa la sustitución hacia atrás
def mi_sustitucion(Ab):
    """Devuelve x como columna, a partir de la aumentada triangular."""
    n = Ab.rows
    x = [None] * n
    # TODO: recorre las filas de abajo hacia arriba y despeja
    return None


mi_x = mi_sustitucion(mi_eliminacion(A, b)) if resultado is not None else None
print("x:", "todavía no" if mi_x is None else mi_x.T.tolist())


In [ ]:
# ✅ Verificación del ejercicio 2 (no modificar)
if mi_x is None:
    print("⏳ El ejercicio 2 sigue pendiente.")
elif verificar(mi_x, A.solve(b)):
    print("✅ Ejercicio 2 correcto:", mi_x.T.tolist())
    print("   Residual A@x - b:", (A @ M.sym[[mi_x[0,0]],[mi_x[1,0]],[mi_x[2,0]]] - b).T.tolist()
          if hasattr(mi_x, "rows") else "—")
else:
    print("❌ Ejercicio 2 incorrecto")
    print("   Se esperaba x =", A.solve(b).T.tolist())
    print("   Recuerda: la suma va sobre las columnas a la DERECHA de la diagonal.")


In [ ]:
# ✏️ EJERCICIO 3 — sin escribir ni un ciclo
#
# Resuelve este sistema con SOLO lo que ya sabes de la biblioteca,
# usando aritmética exacta, y guarda el resultado en `x_facil`.

C3 = M.sym[[4, -2,  1],
           [-2, 4, -2],
           [1, -2,  4]]
d3 = M.sym[[11], [-16], [17]]

x_facil = None      # TODO: una sola expresión
print("x_facil:", "todavía no" if x_facil is None else x_facil)


In [ ]:
# ✅ Verificación del ejercicio 3 (no modificar)
if x_facil is None:
    print("⏳ El ejercicio 3 sigue pendiente.")
else:
    x3 = x_facil if hasattr(x_facil, "rows") else var(x_facil)
    if verificar(x3, C3.solve(d3)):
        print("✅ Ejercicio 3 correcto:", x3.T.tolist())
        print("   ¿Exacto?", es_exacto(x3), "· ¿Residual cero?", (C3 @ x3 - d3).is_zero_matrix)
    else:
        print("❌ Ejercicio 3 incorrecto. Se esperaba:", C3.solve(d3).T.tolist())
        print("   Pista: la biblioteca ya sabe resolver sistemas; solo hay que pedírselo.")


## 8. Caso de ingeniería: un circuito de tres mallas

Un circuito resistivo con tres mallas produce, por la ley de voltajes de
Kirchhoff, un sistema lineal cuya matriz es **simétrica** y con la diagonal
dominante:

$$\begin{bmatrix} R_3+R_1 & -R_1 & 0 \\ -R_1 & R_1+R_2+R_4 & -R_2 \\
0 & -R_2 & R_2+R_5 \end{bmatrix}
\begin{bmatrix} i_1 \\ i_2 \\ i_3 \end{bmatrix}
= \begin{bmatrix} V_1 \\ V_2 \\ V_3 \end{bmatrix}$$

| Resistencia | Valor | Dónde está |
|---|---|---|
| $R_1$ | $2\ \Omega$ | compartida entre mallas 1 y 2 |
| $R_2$ | $4\ \Omega$ | compartida entre mallas 2 y 3 |
| $R_3$ | $6\ \Omega$ | solo malla 1 |
| $R_4$ | $10\ \Omega$ | solo malla 2 |
| $R_5$ | $8\ \Omega$ | solo malla 3 |

Con esas resistencias la matriz queda numérica y pequeña, así que se puede
resolver **exactamente**:


In [ ]:
R1_, R2_, R3_, R4_, R5_ = 2, 4, 6, 10, 8

R = M.sym[[R3_ + R1_, -R1_,               0],
          [-R1_,     R1_ + R2_ + R4_, -R2_],
          [0,       -R2_,              R2_ + R5_]]
V = M.sym[[20], [22], [4]]

print("Matriz del circuito:")
mostrar(R)
print("\n¿Simétrica?", R == R.T, "| ¿det?", R.det())
print("Dominancia diagonal por fila:")
for i in range(3):
    propia = abs(R[i, i])
    otras = sum(abs(R[i, j]) for j in range(3) if j != i)
    print(f"  fila {i+1}: |diagonal| = {propia} vs suma de las otras = {otras}"
          f"  {'✓' if propia > otras else '✗'}")


In [ ]:
# Resolver con el método del cuaderno
i_mallas = gauss_solve(R, V)
print("Corrientes de malla (A):")
mostrar(i_mallas)

# Comprobar contra la biblioteca y contra el residual
print("A.solve(V) :", R.solve(V).T.tolist())
print("¿Coinciden?:", i_mallas == R.solve(V))
print("\nResidual R@i - V:")
mostrar(R @ i_mallas - V)
print("¿Cero exacto?", (R @ i_mallas - V).is_zero_matrix)


In [ ]:
# ¿Cuánto costó? El costo predicho por la fórmula, para n = 3
n = 3
predicho = sp.Rational(2,3)*n**3 + sp.Rational(1,2)*n**2 - sp.Rational(7,6)*n + n**2
print("operaciones predichas para n=3:", predicho)

# Y qué habría pasado con pivoteo (numérico), para comparar
print("con gauss_pivoteo (float):", gauss_pivoteo(R, V).round(12).tolist())
print("exacto                   :", [float(v) for v in i_mallas])


### 8.1 ¿Gauss es el método correcto para este caso?

Sí, y conviene saber **por qué**:

| Criterio | Este caso | Consecuencia |
|---|---|---|
| Tamaño | $n = 3$ | el costo $O(n^3)$ es irrelevante |
| Estructura | densa (casi todos los coeficientes ≠ 0) | no hay atajo que explotar |
| Simetría | sí, $\mathbf{R} = \mathbf{R}^\top$ | se podría usar Cholesky, pero no hace falta |
| Diagonal dominante | sí | converge también por métodos iterativos, pero con $n=3$ no vale la pena |
| Necesitas $\det$ o $i$ | solo las corrientes | Gauss resuelve y ya |

**Cuándo NO usarías Gauss aquí:**

- Si el circuito tuviera **50 mallas en cadena** (matriz tridiagonal), el
  algoritmo de **Thomas** cuesta $O(n)$ en vez de $O(n^3)$: para $n=50$ es unas
  17 veces menos trabajo. Eso se ve en el Cuaderno 04.
- Si el circuito tuviera **miles de mallas** y hubiera que resolverlo para
  muchos vectores $\mathbf{V}$ distintos, conviene **descomposición LU**
  (Cuaderno 03): se paga una vez la factorización y cada $\mathbf{V}$ nuevo
  cuesta solo $O(n^2)$.
- Si la matriz fuera **dispersa y muy grande** (miles de nodos), los
  **iterativos** ganan porque no destruyen los ceros.

La regla corta: **Gauss es el método por omisión para un sistema denso y
chico**. Los otros existen para explotar estructura o para reusar trabajo.


In [ ]:
# Comprobación física: la potencia total disipada debe ser positiva
# P = sum(i_k^2 * R_propia_k)  con las resistencias propias de cada malla
propias = [R3_ + R1_, R1_ + R2_ + R4_, R2_ + R5_]
P = sum(float(i_mallas[k])**2 * propias[k] for k in range(3))
print("Potencia disipada (W):", round(P, 4), "| ¿positiva?", P > 0)

# Y los voltajes de malla que pediría el problema inverso
print("\nSi las corrientes fueran [3,2,1] A, las fuentes serían V =",
      (R @ M.sym[[3],[2],[1]]).T.tolist())


## 9. Preguntas calificables

Ocho preguntas sobre este cuaderno. Cinco de teoría y tres donde escribes el
resultado de **tus** matrices.

### Cómo se contesta

1. Ejecuta la celda de un bloque de preguntas.
2. Elige la opción, o escribe la respuesta (`[[1,2,3],[4,5,6]]` para una matriz).
3. Pulsa **Guardar respuesta**. Sin ese botón no se guarda, aunque ya la veas escrita.
4. Cuando termines, ejecuta **Calificar** (no envía nada).
5. Si ya estás conforme, ejecuta **Enviar**.

| | |
|---|---|
| 👀 Calificar sin enviar | las veces que quieras |
| 📤 Envíos al profesor | **5** como máximo |


In [ ]:
#@title 🧪 Preguntas 1 a 4 · el método y los multiplicadores
pregunta(1)
pregunta(2)
pregunta(3)
pregunta(4)


In [ ]:
#@title 🧪 Preguntas 5 a 8 · con tus matrices
pregunta(5)
pregunta(6)
pregunta(7)
pregunta(8)


In [ ]:
#@title 📝 Calificar (sin enviar) — úsala las veces que quieras
calificar()


In [ ]:
#@title 📤 Enviar la calificación (máximo 5 intentos)
enviar(alumno_id)


In [ ]:
#@title 🔎 Comprobar lo que ya está guardado (no envía nada)
consultar_calificacion()


---
### ✅ ¿Qué sigue?

Ya resuelves sistemas con el método directo más general. En el **Cuaderno 03**
verás cómo **reusar** la eliminación: una vez factorizada $\mathbf{A} =
\mathbf{L}\mathbf{U}$, resolver para un $\mathbf{b}$ nuevo cuesta mucho menos
que empezar de cero.

📖 La teoría completa, en [`docs/metodos/01_eliminacion_gaussiana.md`](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/main/docs/metodos/01_eliminacion_gaussiana.md).
